# CU28 mixed_context - Raw Data Profile

Notebook narrativo de auditoria para el scope `mixed_context`.


## Objetivo

Perfilar los snapshots raw oficiales antes del ETL para mostrar su estructura, cobertura y limitaciones como senales externas/proxy.


## Alcance

Este analisis describe la ruta oficial reproducible `mixed_context`. Las senales externas se tratan como contexto/proxy. Las variables internas de planta siguen siendo sinteticas salvo carga posterior de cliente.


## Inputs

            - `data/raw/external/INE_CPI/`
- `data/raw/external/MAPA_SLAUGHTER_MAPA/`
- `data/raw/external/MAPA_PRICES_OM/`
- `data/raw/external/*/source_manifest.json`


## Outputs esperados

            - `reports/tables/eda/raw_file_inventory__mixed_context.csv`
- `reports/tables/eda/raw_data_quality__mixed_context.csv`
- `reports/tables/eda/raw_temporal_ranges__mixed_context.csv`
- `reports/figures/eda/raw_file_sizes__mixed_context.png`
- `reports/figures/eda/raw_record_counts__mixed_context.png`
- `reports/figures/eda/raw_temporal_coverage__mixed_context.png`
- `reports/figures/eda/raw_missing_values__mixed_context.png`


## Limitaciones

Este notebook documenta evidencia reproducible del pipeline oficial, pero no sustituye la revision de codigo, la auditoria de datos de origen ni una certificacion operacional de planta.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.reproducibility.notebook_support import (
    detect_temporal_columns,
    ensure_eda_dirs,
    execution_metadata,
    first_valid_temporal_range,
    load_source_manifests,
    load_tabular_file,
    parse_markdown_table,
    print_frame,
    print_series,
    project_root,
    read_json,
    relative_to_root,
    save_figure,
    save_table,
    sha256_file,
)


In [ ]:
NOTEBOOK_NAME = "01_raw_data_profile.ipynb"
PROJECT_ROOT = project_root()
SCOPE = globals().get("scope", "mixed_context")
REPORT_DIRS = ensure_eda_dirs()
META = execution_metadata(SCOPE)
FIGURES = []
TABLES = []
print(json.dumps(META, indent=2))


## Carga de datos

Las siguientes celdas cargan los artefactos de entrada y muestran verificaciones intermedias antes de producir tablas y graficas.


In [ ]:
raw_root = PROJECT_ROOT / "data" / "raw" / "external"
source_manifests = load_source_manifests(raw_root)
if not source_manifests:
    raise FileNotFoundError(f"No source manifests found under {raw_root}")
raw_inventory_rows = []
for manifest in source_manifests:
    for raw_file in manifest.get("raw_files", []):
        file_path = PROJECT_ROOT / raw_file["path"]
        raw_inventory_rows.append(
            {
                "source_id": manifest["source_id"],
                "path": raw_file["path"],
                "size_bytes": file_path.stat().st_size if file_path.exists() else raw_file.get("size_bytes"),
                "sha256": sha256_file(file_path) if file_path.exists() else raw_file.get("sha256"),
                "access_date": manifest["access_date"],
                "retrieval_method": manifest["retrieval_method"],
            }
        )
if not raw_inventory_rows:
    raise FileNotFoundError(f"No raw files declared in source manifests under {raw_root}")
raw_inventory = pd.DataFrame(raw_inventory_rows).sort_values(["source_id", "path"]).reset_index(drop=True)
print_frame("Raw inventory", raw_inventory, rows=20)
display(raw_inventory)


In [ ]:
profiles = []
previews = {}
tail_previews = {}
for row in raw_inventory.itertuples(index=False):
    file_path = PROJECT_ROOT / row.path
    frame = load_tabular_file(file_path)
    temporal = first_valid_temporal_range(frame)
    previews[row.path] = frame.head(3)
    tail_previews[row.path] = frame.tail(3)
    profiles.append(
        {
            "source_id": row.source_id,
            "path": row.path,
            "row_count": int(len(frame)),
            "column_count": int(len(frame.columns)),
            "columns": ", ".join(str(column) for column in frame.columns[:12]),
            "dtypes": ", ".join(f"{column}:{dtype}" for column, dtype in frame.dtypes.astype(str).items()),
            "missing_pct_mean": float(frame.isna().mean().mean()),
            "duplicate_rows": int(frame.duplicated().sum()),
            "temporal_column": temporal["column"],
            "date_min": temporal["date_min"],
            "date_max": temporal["date_max"],
        }
    )
raw_quality = pd.DataFrame(profiles).sort_values(["source_id", "path"]).reset_index(drop=True)
print_frame("Raw profile", raw_quality, rows=20)
display(raw_quality[["source_id", "path", "row_count", "column_count", "temporal_column", "date_min", "date_max"]])


## Inspeccion inicial

En esta fase se comprueban shape, columnas, tipos, nulos y duplicados. La lectura usa el primer sheet reproducible de cada snapshot tabular.


In [ ]:
shape_summary = raw_quality[["source_id", "path", "row_count", "column_count"]].copy()
print_frame("Shape by raw file", shape_summary, rows=20)
display(shape_summary)


In [ ]:
column_and_dtype_summary = raw_quality[["source_id", "path", "columns", "dtypes"]].copy()
print_frame("Columns and dtypes", column_and_dtype_summary, rows=20)
display(column_and_dtype_summary.head(10))


In [ ]:
quality_summary = raw_quality[["source_id", "path", "missing_pct_mean", "duplicate_rows"]].copy()
quality_summary["missing_pct_mean"] = quality_summary["missing_pct_mean"].round(4)
print_frame("Missing values and duplicates", quality_summary, rows=20)
display(quality_summary)


In [ ]:
for path, preview in previews.items():
    print(f"First rows for {path}")
    print(preview.to_string(index=False))
    print("")


In [ ]:
for path, preview in tail_previews.items():
    print(f"Last rows for {path}")
    print(preview.to_string(index=False))
    print("")


In [ ]:
temporal_ranges = raw_quality[["source_id", "path", "temporal_column", "date_min", "date_max"]].copy()
print_frame("Temporal range by raw file", temporal_ranges, rows=20)
display(temporal_ranges)


## Interpretacion intermedia

Los snapshots raw son insumos contextuales externos. No representan inventario observado, compras observadas ni produccion observada de una planta concreta.


In [ ]:
size_by_source = raw_inventory.groupby("source_id", as_index=False)["size_bytes"].sum().sort_values("size_bytes", ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(size_by_source["source_id"], size_by_source["size_bytes"], color="#355c7d")
ax.set_title("Raw size by source")
ax.set_ylabel("bytes")
FIGURES.append(save_figure(fig, "raw_file_sizes__mixed_context.png"))
plt.close(fig)
print_frame("Size by source", size_by_source)


### Interpretacion de la figura

El tamano por fuente ayuda a entender el peso del blob y confirma que el repositorio no necesita contener datos masivos si el empaquetado externo esta bien trazado.


In [ ]:
records_by_source = raw_quality.groupby("source_id", as_index=False)["row_count"].sum().sort_values("row_count", ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(records_by_source["source_id"], records_by_source["row_count"], color="#c06c84")
ax.set_title("Raw record count by source")
ax.set_ylabel("rows")
FIGURES.append(save_figure(fig, "raw_record_counts__mixed_context.png"))
plt.close(fig)
print_frame("Records by source", records_by_source)


### Interpretacion de la figura

El conteo de filas ilustra que las fuentes aportan granularidades distintas y que el ETL posterior debe armonizar frecuencia y estructura.


In [ ]:
temporal_plot_df = temporal_ranges.dropna(subset=["date_min", "date_max"]).copy()
temporal_plot_df["date_min"] = pd.to_datetime(temporal_plot_df["date_min"])
temporal_plot_df["date_max"] = pd.to_datetime(temporal_plot_df["date_max"])
fig, ax = plt.subplots(figsize=(10, 4))
for idx, row in temporal_plot_df.reset_index(drop=True).iterrows():
    ax.hlines(idx, row["date_min"], row["date_max"], linewidth=6, color="#6c5b7b")
ax.set_yticks(range(len(temporal_plot_df)))
ax.set_yticklabels(temporal_plot_df["source_id"])
ax.set_title("Temporal coverage by raw file")
ax.set_xlabel("date")
FIGURES.append(save_figure(fig, "raw_temporal_coverage__mixed_context.png"))
plt.close(fig)


In [ ]:
missing_by_file = raw_quality[["path", "missing_pct_mean"]].sort_values("missing_pct_mean", ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(missing_by_file["path"], missing_by_file["missing_pct_mean"], color="#f67280")
ax.set_title("Average missing rate by raw file")
ax.set_ylabel("missing_pct_mean")
ax.tick_params(axis="x", rotation=75)
FIGURES.append(save_figure(fig, "raw_missing_values__mixed_context.png"))
plt.close(fig)
print_frame("Missing rate by raw file", missing_by_file, rows=20)


## Hallazgos parciales

- El inventario raw queda identificado por ruta, hash y metodo de obtencion.
- La cobertura temporal es heterogenea y confirma la necesidad de un ETL que armonice frecuencia.
- Las hojas tabulares no contienen variables internas observadas de planta.


In [ ]:
TABLES.append(save_table(raw_inventory, "raw_file_inventory__mixed_context.csv"))
TABLES.append(save_table(raw_quality, "raw_data_quality__mixed_context.csv"))
TABLES.append(save_table(temporal_ranges, "raw_temporal_ranges__mixed_context.csv"))
RESULT = {
    "notebook": NOTEBOOK_NAME,
    "tables": TABLES,
    "figures": FIGURES,
    "findings": [
        "Raw files are external contextual signals and do not contain internal observed plant variables.",
        "Temporal coverage differs by source and motivates the weekly harmonisation step.",
        "Each raw file remains traceable through size, hash and access metadata.",
    ],
    "limitations": [
        "Spreadsheet profiling reads the first sheet only, which is the defended audit entrypoint for these snapshots.",
    ],
}
print(json.dumps(RESULT, indent=2))


## Limitaciones

El perfil raw no sustituye la inspeccion sectorial de contenido. Su objetivo es dejar constancia reproducible de estructura, completitud y cobertura antes del ETL.


## Concluson final

Los raw oficiales quedan caracterizados como snapshots externos/proxy listos para transformacion, sin insinuar que sean historicos internos de aprovisionamiento o produccion.
